In [3]:
import os
from pathlib import Path

import torch, yaml
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

if Path.cwd().name == "notebooks":
    os.chdir("..")                                   # run from the repo root
ROOT = Path.cwd()
CFG = yaml.safe_load((ROOT / "configs" / "config.yaml").read_text())
MAX_IN, G = CFG["data"]["max_source_len"], CFG["baseline"]["decoding"]
MODEL_DIR = ROOT / "outputs" / "model"

# Use the local model if it exists, otherwise download it from Hugging Face (first use only, then cached).
MODEL_SRC = MODEL_DIR if (MODEL_DIR / "config.json").exists() else CFG["model_hub"]
print("model:", "local outputs/model" if MODEL_SRC == MODEL_DIR else f"Hugging Face ({MODEL_SRC})")

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_SRC)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_SRC).to(device).eval()


def summarize(abstract):
    abstract = " ".join(abstract.split())            
    n_tokens = len(tok(abstract)["input_ids"])
    batch = tok(abstract, max_length=MAX_IN, truncation=True, return_tensors="pt").to(device)
    with torch.inference_mode():
        ids = model.generate(**batch, **G)      
    return tok.decode(ids[0], skip_special_tokens=True).strip(), n_tokens


print("model ready on", device)

model: local outputs/model
model ready on cuda


In [4]:
import ipywidgets as widgets
from IPython.display import display

box = widgets.Textarea(placeholder="Paste an abstract here...",
                       layout=widgets.Layout(width="100%", height="180px"))
button = widgets.Button(description="Summarize", button_style="primary")
out = widgets.Output()


def on_click(_):
    with out:
        out.clear_output()
        text = box.value.strip()
        if not text:
            print("Paste an abstract first.")
            return
        summary, n_tokens = summarize(text)
        print("TLDR:", summary)
        if n_tokens > MAX_IN:
            print(f"(note: the abstract has {n_tokens} tokens; only the first {MAX_IN} were read)")


button.on_click(on_click)
display(box, button, out)

Textarea(value='', layout=Layout(height='180px', width='100%'), placeholder='Paste an abstract here...')

Button(button_style='primary', description='Summarize', style=ButtonStyle())

Output()